In [1]:
class WakeupException(message: String) : RuntimeException(message)

# Forenklet implementasjon av KafkaConsumer og wakeup()

Demonstrerer blokkering av tråden i jvm og samtidig bruk av coroutines i Kotlin.

In [1]:
import java.util.concurrent.atomic.AtomicBoolean

class FakeKafkaConsumer(private val pollBlockTimeMs: Long = 10_000) {
    // Mirrors KafkaConsumer's internal wakeup trigger
    private val wakeupTriggered = AtomicBoolean(false)

    // We need to track the polling thread so wakeup() can interrupt it
    @Volatile
    private var pollingThread: Thread? = null

    /**
     * Simulates KafkaConsumer.poll(Duration).
     *
     * Blocks the calling thread for [pollBlockTimeMs] milliseconds — just like
     * real poll() blocks while waiting for records from the broker.
     *
     * If wakeup() was called, throws WakeupException immediately.
     */
    fun poll(): List<String> {
        // Real KafkaConsumer checks the wakeup flag at the start of poll()
        if (wakeupTriggered.getAndSet(false)) {
            throw WakeupException("Consumer woke up!")
        }

        pollingThread = Thread.currentThread()
        println("    [FakeConsumer] poll() blocking for ${pollBlockTimeMs}ms...")
        try {
            // This is the critical part: Thread.sleep blocks the JVM thread.
            // Kotlin coroutine cancellation CANNOT interrupt this.
            // Only Thread.interrupt() (triggered by wakeup) can break out.
            Thread.sleep(pollBlockTimeMs)
        } catch (_: InterruptedException) {
            // wakeup() interrupted us mid-sleep — check the flag
            if (wakeupTriggered.getAndSet(false)) {
                throw WakeupException("Consumer woke up during poll!")
            }
        } finally {
            pollingThread = null
        }

        println("    [FakeConsumer] poll() returned records")
        return listOf("record-1", "record-2")
    }

    /**
     * Simulates KafkaConsumer.wakeup().
     *
     * Thread-safe. Can be called from ANY thread. Makes the next (or current)
     * poll() call throw WakeupException instead of blocking.
     */
    fun wakeup() {
        println("    [FakeConsumer] wakeup() called — will interrupt poll()")
        wakeupTriggered.set(true)
        // Interrupt the thread stuck in poll() so it wakes up immediately
        pollingThread?.interrupt()
    }

    fun close() {
        println("    [FakeConsumer] closed")
    }
}

org.jetbrains.kotlinx.jupyter.exceptions.ReplCompilerException: at Cell In[1], line 22, column 19: Unresolved reference: WakeupException
at Cell In[1], line 35, column 23: Unresolved reference: WakeupException

In [11]:
import kotlinx.coroutines.CoroutineName
import kotlinx.coroutines.Dispatchers
import kotlinx.coroutines.coroutineScope
import kotlinx.coroutines.delay
import kotlinx.coroutines.isActive
import kotlinx.coroutines.launch
import kotlinx.coroutines.runBlocking
import kotlinx.coroutines.withContext
import kotlin.coroutines.cancellation.CancellationException

class BrokenConsumerLoop {
    private val consumer = FakeKafkaConsumer(pollBlockTimeMs = 5_000)

    suspend fun run() = coroutineScope {
        val startTime = System.currentTimeMillis()

        // Launch the consumer loop on Dispatchers.IO — same as our real code
        val job = launch(Dispatchers.IO + CoroutineName("broken-consumer")) {
            println("[Consumer] Started consumer loop")
            listen()
        }

        // Simulate the application running for 1 second, then requesting shutdown
        delay(1.seconds)
        val cancelTime = System.currentTimeMillis() - startTime
        println("\n[Shutdown] Cancelling consumer job at ${cancelTime}ms...")

        // This is what ApplicationStop(Preparing) does: job.cancel()
        // But we have NO consumer.wakeup() call here
        job.cancel()

        // Wait for the job to actually finish
        job.join()
        val totalTime = System.currentTimeMillis() - startTime
        println("[Shutdown] Consumer job completed after ${totalTime}ms")
        println("[Shutdown] ⚠️  Expected ~1s, but took ~${totalTime / 1000}s because poll() blocked!")
    }

    /**
     * The broken consumer loop.
     *
     * After job.cancel():
     *   1. isActive becomes false immediately — but we can't check it while blocked in poll()
     *   2. poll() keeps blocking on Thread.sleep for the full duration
     *   3. When poll() finally returns, the coroutine runtime tries to throw CancellationException
     *   4. Our catch(e: Exception) catches it and prints "Error listening" — oops!
     *   5. Loop checks isActive → false → exits (only because cancel has already happened)
     *      ...but if there were a timing issue, the swallowed CancellationException
     *      could cause the loop to continue indefinitely.
     */
    private suspend fun listen() = withContext(Dispatchers.IO) {
        try {
            while (isActive) {
                try {
                    println("  [Consumer] Calling poll()...")
                    val records = consumer.poll()  // ← BLOCKS here, cannot be cancelled!

                    println("  [Consumer] Processing ${records.size} records...")
                    records.forEach { println("    Processed: $it") }
                    // processRecords(records) // contains suspension point
                } catch (e: Exception) {
                    // 🐛 BUG: This catches CancellationException too!
                    // CancellationException is a subclass of IllegalStateException → Exception.
                    // By catching it, we break structured concurrency.
                    println("  [Consumer] ❌ Caught exception: ${e::class.simpleName}: ${e.message}")
                    println("  [Consumer] ❌ Was this a CancellationException? ${e is CancellationException}")
                    if (e is CancellationException) {
                        println("  [Consumer] ❌ We just swallowed a CancellationException!")
                        println("  [Consumer] ❌ The coroutine should have stopped, but catch(Exception) ate it.")
                    }
                }
            }
        } catch (_: CancellationException) {
            // This will never be reached in the current code, because CancellationException is caught above.
            println("  [Consumer] ✅ CancellationException propagated correctly — shutting down")
        } finally {
            consumer.close()
            println("[Consumer] Exited loop (isActive=$isActive)")
        }
    }

    // Denne har et suspension point (coroutineScope), som vil kaste exception dersom parent jobben er kansellert
    suspend fun processRecords(records: List<String>) = coroutineScope {
        records.forEach { println("    Processed: $it") }
    }
}
runBlocking {
    BrokenConsumerLoop().run()
}


[Consumer] Started consumer loop
  [Consumer] Calling poll()...
    [FakeConsumer] poll() blocking for 5000ms...

[Shutdown] Cancelling consumer job at 1005ms...
    [FakeConsumer] poll() returned records
  [Consumer] Processing 2 records...
    Processed: record-1
    Processed: record-2
    Processed: record-1
    Processed: record-2
  [Consumer] ❌ Caught exception: JobCancellationException: StandaloneCoroutine was cancelled
  [Consumer] ❌ Was this a CancellationException? true
  [Consumer] ❌ We just swallowed a CancellationException!
  [Consumer] ❌ The coroutine should have stopped, but catch(Exception) ate it.
    [FakeConsumer] closed
[Consumer] Exited loop (isActive=false)
[Shutdown] Consumer job completed after 5015ms
[Shutdown] ⚠️  Expected ~1s, but took ~5s because poll() blocked!


In [5]:
import kotlin.coroutines.cancellation.CancellationException

inline fun <R, reified E> runCatchingOrThrow(block: () -> R): Result<R> = try {
    Result.success(block())
} catch (e: Exception) {
    if (e is E) {
        println("Re-throwing cancellation...")
        throw e
    }
    Result.failure(e)
}

inline fun <R> runCatchingCancellable(block: () -> R): Result<R> = runCatchingOrThrow<R, CancellationException> { block() }

In [9]:
import kotlinx.coroutines.awaitCancellation
import kotlinx.coroutines.job

class FixedConsumerLoop {
    private val consumer = FakeKafkaConsumer(pollBlockTimeMs = 5_000)

    fun run() = runBlocking {
        val startTime = System.currentTimeMillis()

        val job = launch(Dispatchers.IO + CoroutineName("fixed-consumer")) {
            println("[Consumer] Started consumer loop")
            listen()
        }

        delay(1.seconds)
        val cancelTime = System.currentTimeMillis() - startTime
        println("\n[Shutdown] Cancelling consumer job at ${cancelTime}ms...")

        // The fix: cancel AND wakeup. In LeaderKafkaConsumerTask this happens
        // automatically via invokeOnCompletion { kafkaConsumer.wakeup() }
        job.cancel()
        // No need to manually call consumer.wakeup() here — it's registered
        // via invokeOnCompletion inside listen() (see below)

        job.join()
        val totalTime = System.currentTimeMillis() - startTime
        println("[Shutdown] Consumer job completed after ${totalTime}ms")
        println("[Shutdown] ✅ Shutdown took ~${totalTime / 1000}s — poll() was interrupted immediately!")
    }

    /**
     * The fixed consumer loop.
     *
     * Key differences from the broken version:
     *   1. invokeOnCompletion { consumer.wakeup() } — registered at startup
     *      When the job is cancelled, wakeup() fires immediately (from the cancelling thread).
     *      This interrupts poll() mid-block, making it throw WakeupException.
     *
     *   2. runCatchingCancellable { ... } instead of try/catch(Exception)
     *      This catches all exceptions EXCEPT CancellationException.
     *      CancellationException is re-thrown, preserving structured concurrency.
     *
     *   3. WakeupException is handled explicitly to break out of the loop.
     */
    private suspend fun listen() = withContext(Dispatchers.IO) {
        launch {
            try {
                awaitCancellation()
            } finally {
                println("  [Consumer] wakeupJob cancelled → calling consumer.wakeup()")
                consumer.wakeup()
            }
        }

        try {
            while (isActive) {
                // runCatchingCancellable: like runCatching, but re-throws CancellationException
                // Source: no.nav.syfo.application.exception.Functions.kt
                //
                // Without this (plain try/catch or runCatching), CancellationException would be
                // caught and treated as "just another error", breaking structured concurrency.
                runCatchingCancellable {
                    println("  [Consumer] Calling poll()...")
                    val records = consumer.poll()  // ← Can now be interrupted by wakeup()!

                    println("  [Consumer] Processing ${records.size} records...")
                    records.forEach { println("    Processed: $it") }
                }.getOrElse { error ->
                    if (error is WakeupException) {
                        // WakeupException means we were asked to stop — break the loop
                        println("  [Consumer] ✅ Got WakeupException — exiting loop cleanly")
                        return@withContext
                    }
                    // Any other error: log and retry (real code has delay + resubscribe)
                    println("  [Consumer] Recoverable error: ${error.message}")
                }
            }
        } catch (_: CancellationException) {
            // This is the normal cancellation path — structured concurrency working as designed
            println("  [Consumer] ✅ CancellationException propagated correctly — shutting down")
        } finally {
            consumer.close()
            println("[Consumer] Exited loop cleanly")
        }
    }
}


FixedConsumerLoop().run()

[Consumer] Started consumer loop
  [Consumer] Calling poll()...
    [FakeConsumer] poll() blocking for 5000ms...

[Shutdown] Cancelling consumer job at 1005ms...
  [Consumer] wakeupJob cancelled → calling consumer.wakeup()
    [FakeConsumer] wakeup() called — will interrupt poll()
  [Consumer] ✅ Got WakeupException — exiting loop cleanly
    [FakeConsumer] closed
[Consumer] Exited loop cleanly
[Shutdown] Consumer job completed after 1008ms
[Shutdown] ✅ Shutdown took ~1s — poll() was interrupted immediately!
